# Protein Stability ΔΔG — v4 (All Fixes Applied)

**Root causes fixed:**
1. ✅ Protein-weighted sampling (1PGA no longer dominates)
2. ✅ Protein-level validation split (no leaking)
3. ✅ Outlier clipping (ΔΔG capped to [-8, 6])
4. ✅ Correlation-aware loss (Huber + Pearson penalty)
5. ✅ Larger ESM-2 (t12_35M → 480D instead of 320D)
6. ✅ Mutation-focused delta ESM at mutation site only

**Upload:** Only `S8754.csv` and `S669.csv` in `ProteinStability/data/raw/`

In [ ]:
# Cell 1: Setup
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.5.0+cu121.html -q
!pip install transformers -q

import os, json, time, copy, gc, requests, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.utils import softmax
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cdist
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Kaggle - no drive mount needed

BASE = "/kaggle/working"
RAW_DIR = "/kaggle/input/protein-stability-data"
RESULTS_DIR = "/kaggle/working/results_v4"
PDB_DIR = "/kaggle/working/pdb_cache"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PDB_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

EDGE_CUTOFF = 12.0
DDG_CLIP_MIN, DDG_CLIP_MAX = -8.0, 6.0  # FIX 3: clip outliers
AA_3TO1 = {'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F','GLY':'G',
           'HIS':'H','ILE':'I','LYS':'K','LEU':'L','MET':'M','ASN':'N',
           'PRO':'P','GLN':'Q','ARG':'R','SER':'S','THR':'T','VAL':'V',
           'TRP':'W','TYR':'Y','MSE':'M'}
AA_LIST = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}

# Physicochemical properties for explicit mutation encoding (FIX: suggestion 4)
HYDRO = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,
         'I':4.5,'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,
         'R':-4.5,'S':-0.8,'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3}
VOLUME = {'A':88.6,'C':108.5,'D':111.1,'E':138.4,'F':189.9,'G':60.1,'H':153.2,
          'I':166.7,'K':168.6,'L':166.7,'M':162.9,'N':114.1,'P':112.7,'Q':143.8,
          'R':173.4,'S':89.0,'T':116.1,'V':140.0,'W':227.8,'Y':193.6}
CHARGE = {'D':-1,'E':-1,'K':1,'R':1,'H':0.5}

print("✅ Setup complete!")

In [ ]:
# Cell 2: Load Data & ESM-2 (larger model: t12_35M → 480D)

train_df = pd.read_csv(os.path.join(RAW_DIR, 'S8754.csv'))
test_df = pd.read_csv(os.path.join(RAW_DIR, 'S669.csv'))

# FIX 3: Clip extreme outliers in training
n_clipped = ((train_df['ddG'] < DDG_CLIP_MIN) | (train_df['ddG'] > DDG_CLIP_MAX)).sum()
train_df['ddG'] = train_df['ddG'].clip(DDG_CLIP_MIN, DDG_CLIP_MAX)
print(f"Train: {len(train_df)} mutations ({n_clipped} DDG values clipped to [{DDG_CLIP_MIN}, {DDG_CLIP_MAX}])")
print(f"Test: {len(test_df)} mutations (untouched)")

# Collect unique sequences (both WT and MUT)
all_seqs = set()
for df in [train_df, test_df]:
    all_seqs.update(df['wt_seq'].tolist())
    all_seqs.update(df['mut_seq'].tolist())
print(f"Unique sequences: {len(all_seqs)}")

# FIX 5: Larger ESM-2 model (t12_35M → 480D)
from transformers import EsmTokenizer, EsmModel
ESM_NAME = "facebook/esm2_t12_35M_UR50D"
ESM_DIM = 480
print(f"\nLoading {ESM_NAME} ({ESM_DIM}D per residue)...")
esm_tok = EsmTokenizer.from_pretrained(ESM_NAME)
esm_mdl = EsmModel.from_pretrained(ESM_NAME).to(device)
esm_mdl.eval()
print("ESM-2 loaded!")

seq2emb = {}
print(f"Extracting embeddings for {len(all_seqs)} sequences...")
t0 = time.time()
with torch.no_grad():
    for i, seq in enumerate(sorted(all_seqs)):
        inp = esm_tok(seq[:1022], return_tensors='pt', padding=False,
                      truncation=True, max_length=1024).to(device)
        out = esm_mdl(**inp)
        seq2emb[seq] = out.last_hidden_state[0, 1:-1, :].cpu()
        if (i+1) % 100 == 0:
            eta = (time.time()-t0)/(i+1)*(len(all_seqs)-i-1)/60
            print(f"  [{i+1}/{len(all_seqs)}] ETA: {eta:.1f}min")

print(f"Done in {(time.time()-t0)/60:.1f} min")
del esm_mdl, esm_tok; torch.cuda.empty_cache(); gc.collect()
print("ESM-2 freed")

In [ ]:
# Cell 3: Build Graphs (full protein + mutation-focused features)

def download_pdb(pdb_id):
    path = os.path.join(PDB_DIR, f"{pdb_id}.pdb")
    if os.path.exists(path): return path
    try:
        r = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30)
        if r.status_code == 200:
            with open(path, 'w') as f: f.write(r.text)
            return path
    except: pass
    return None

def get_ca(pdb_path, chain):
    res = []; seen = set()
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith('ATOM') or line[12:16].strip()!='CA': continue
            if line[21].strip()!=chain: continue
            rs=int(line[22:26].strip()); ic=line[26].strip()
            if (rs,ic) in seen: continue
            seen.add((rs,ic))
            aa=AA_3TO1.get(line[17:20].strip(),'X')
            if aa=='X': continue
            res.append({'aa':aa,'coords':np.array([float(line[30:38]),float(line[38:46]),float(line[46:54])])})
    return res

def align(pdb_res, csv_seq):
    ps = ''.join(r['aa'] for r in pdb_res)
    idx = csv_seq.find(ps)
    if idx >= 0: return [(idx+i, pdb_res[i]) for i in range(len(pdb_res))]
    bo, bn = 0, 0
    for o in range(max(0, len(csv_seq)-len(ps)+1)):
        n = sum(1 for i in range(min(len(ps),len(csv_seq)-o)) if ps[i]==csv_seq[o+i])
        if n > bn: bn, bo = n, o
    if bn >= len(ps)*0.8:
        return [(bo+i, pdb_res[i]) for i in range(min(len(ps),len(csv_seq)-bo)) if ps[i]==csv_seq[bo+i]]
    return None


def build_graph_v4(row, seq2emb):
    """Build graph with all v4 fixes."""
    parts = row['name'].split('_')
    if parts[0]!='rcsb' or len(parts)<6: return None
    pdb_id, chain = parts[1].upper(), parts[2]
    wt_aa, mut_aa = parts[3][0], parts[3][-1]
    wt_seq, mut_seq = row['wt_seq'], row['mut_seq']
    ddG = row['ddG']
    pH = float(parts[4]) if len(parts)>4 else 7.0
    temp = float(parts[5]) if len(parts)>5 else 25.0

    # Find mutation position
    mut_seq_idx = None
    for i in range(min(len(wt_seq), len(mut_seq))):
        if wt_seq[i] != mut_seq[i]: mut_seq_idx = i; break
    if mut_seq_idx is None: return None

    pdb_path = download_pdb(pdb_id)
    if not pdb_path: return None
    pdb_res = get_ca(pdb_path, chain)
    if len(pdb_res) < 10: return None
    alignment = align(pdb_res, wt_seq)
    if not alignment or len(alignment) < 10: return None
    if wt_seq not in seq2emb or mut_seq not in seq2emb: return None

    wt_emb = seq2emb[wt_seq]   # (L, 480)
    mut_emb = seq2emb[mut_seq]  # (L, 480)

    # Find mutation node
    all_coords = np.array([a[1]['coords'] for a in alignment])
    mut_node = None
    for ni, (si, _) in enumerate(alignment):
        if si == mut_seq_idx: mut_node = ni; break
    if mut_node is None: return None

    n = len(alignment)
    mut_coords = all_coords[mut_node]
    dists_to_mut = np.linalg.norm(all_coords - mut_coords, axis=1)

    # FIX 4 (suggestion 4): Explicit mutation encoding
    delta_hydro = (HYDRO.get(mut_aa, 0) - HYDRO.get(wt_aa, 0)) / 9.0  # normalize
    delta_vol = (VOLUME.get(mut_aa, 130) - VOLUME.get(wt_aa, 130)) / 150.0
    delta_charge = CHARGE.get(mut_aa, 0) - CHARGE.get(wt_aa, 0)

    # FIX 6: Delta ESM at mutation position only
    if mut_seq_idx < wt_emb.shape[0] and mut_seq_idx < mut_emb.shape[0]:
        esm_delta_at_mut = (mut_emb[mut_seq_idx] - wt_emb[mut_seq_idx]).numpy()  # (480,)
    else:
        esm_delta_at_mut = np.zeros(ESM_DIM)

    # Build node features
    node_feats = []
    for ni, (si, pr) in enumerate(alignment):
        aa_idx = AA_TO_IDX.get(pr['aa'], 0)
        is_mut = 1.0 if ni == mut_node else 0.0
        d_to_mut = dists_to_mut[ni] / 20.0  # normalize by ~max expected distance

        # WT ESM-2 at this position
        wt_esm = wt_emb[si].numpy() if si < wt_emb.shape[0] else np.zeros(ESM_DIM)

        # Features: [aa_idx, is_mut, wt_aa_idx, mut_aa_idx, d_to_mut,
        #            pH, temp, delta_hydro, delta_vol, delta_charge,
        #            wt_esm(480), esm_delta_at_mut(480)] = 970D
        feat = np.concatenate([
            [aa_idx, is_mut, AA_TO_IDX.get(wt_aa,0), AA_TO_IDX.get(mut_aa,0),
             d_to_mut, pH/14, temp/100, delta_hydro, delta_vol, delta_charge],
            wt_esm,
            esm_delta_at_mut,  # SAME delta vector broadcast to all nodes
        ])
        node_feats.append(feat)

    node_feats = np.array(node_feats, dtype=np.float32)

    # Build edges
    dm = cdist(all_coords, all_coords)
    src, dst, ed, edir = [], [], [], []
    for i in range(n):
        for j in range(n):
            if i==j: continue
            d = dm[i,j]
            if d <= EDGE_CUTOFF:
                src.append(i); dst.append(j)
                ed.append(d/EDGE_CUTOFF)
                edir.append((all_coords[j]-all_coords[i])/(d+1e-8))
    if not src: return None

    ei = torch.tensor([src,dst], dtype=torch.long)
    e_d = torch.tensor(ed, dtype=torch.float32)
    ea = torch.cat([e_d.unsqueeze(-1), torch.tensor(np.array(edir),dtype=torch.float32)], 1)

    g = Data(x=torch.tensor(node_feats, dtype=torch.float32),
             edge_index=ei, edge_attr=ea, edge_dist=e_d,
             y=torch.tensor([ddG], dtype=torch.float32),
             mut_idx=torch.tensor([mut_node], dtype=torch.long),
             pos=torch.tensor(all_coords, dtype=torch.float32))
    g.pdb_id = pdb_id; g.chain = chain
    return g

print("✅ Graph builder ready (full protein + mutation-focused delta ESM)")

In [ ]:
# Cell 4: Build All Graphs + Compute Protein Weights

def build_all(df, name):
    graphs, failed = [], 0; t0 = time.time()
    for i, (_, row) in enumerate(df.iterrows()):
        g = build_graph_v4(row, seq2emb)
        if g: graphs.append(g)
        else: failed += 1
        if (i+1)%500==0:
            eta=(time.time()-t0)/(i+1)*(len(df)-i-1)/60
            print(f"  [{i+1}/{len(df)}] ok={len(graphs)} fail={failed} ETA={eta:.1f}m")
    print(f"  {name}: {len(graphs)} graphs, {failed} failed, {(time.time()-t0)/60:.1f}min")
    return graphs

print("Building training graphs...")
train_graphs = build_all(train_df, 'train')
print("\nBuilding test graphs...")
test_graphs = build_all(test_df, 'test')

del seq2emb; gc.collect()

in_ch = train_graphs[0].x.shape[1]
print(f"\nNode features: {in_ch}D")

# FIX 1: Compute per-protein sample weights
# Weight = 1/sqrt(n_mutations_for_protein)
from collections import Counter
pdb_counts = Counter(g.pdb_id for g in train_graphs)
for g in train_graphs:
    g.sample_weight = 1.0 / math.sqrt(pdb_counts[g.pdb_id])

print(f"\nProtein weighting applied:")
top5 = pdb_counts.most_common(5)
for pdb, cnt in top5:
    w = 1.0 / math.sqrt(cnt)
    print(f"  {pdb}: {cnt} mutations → weight={w:.3f} (was 1.0)")

# FIX 2: Protein-level validation split
train_pdbs = list(set(g.pdb_id for g in train_graphs))
np.random.seed(42)
np.random.shuffle(train_pdbs)
n_val_pdbs = max(5, len(train_pdbs) // 10)
val_pdb_set = set(train_pdbs[:n_val_pdbs])

actual_train = [g for g in train_graphs if g.pdb_id not in val_pdb_set]
val_graphs = [g for g in train_graphs if g.pdb_id in val_pdb_set]
print(f"\nProtein-level split (NO leak):")
print(f"  Train: {len(actual_train)} mutations from {len(set(g.pdb_id for g in actual_train))} proteins")
print(f"  Val: {len(val_graphs)} mutations from {len(val_pdb_set)} proteins")
print(f"  Overlap: 0 proteins ✓")

print(f"\n✅ {len(train_graphs)} train + {len(test_graphs)} test graphs ready!")

In [ ]:
# Cell 5: Model (v4 — additive distance like v2, with all data fixes)

class DistKernel(nn.Module):
    def __init__(self, n_k=3):
        super().__init__()
        self.log_s = nn.Parameter(torch.log(torch.tensor([0.10, 0.25, 0.50][:n_k])))
        self.w = nn.Parameter(torch.ones(n_k)/n_k)
        self.n_k = n_k
    def forward(self, d):
        s = torch.exp(self.log_s)
        w = F.softmax(self.w, 0)
        return (torch.exp(-d.unsqueeze(-1)**2/(2*s**2+1e-8))*w).sum(-1)

class DistGATLayer(MessagePassing):
    def __init__(self, in_ch, out_ch, heads=4, n_k=3, e_dim=4,
                 drop=0.2, concat=True, use_dist=True):
        super().__init__(aggr='add', node_dim=0)
        self.heads, self.out_ch, self.concat = heads, out_ch, concat
        self.drop, self.use_dist = drop, use_dist
        self.W = nn.Linear(in_ch, heads*out_ch, bias=False)
        self.a_s = nn.Parameter(torch.zeros(1, heads, out_ch))
        self.a_d = nn.Parameter(torch.zeros(1, heads, out_ch))
        self.ep = nn.Sequential(nn.Linear(e_dim, heads), nn.LeakyReLU(0.2))
        if use_dist:
            self.dk = DistKernel(n_k)
            self.gamma = nn.Parameter(torch.tensor(1.0))
        self.ln = nn.LayerNorm(heads*out_ch if concat else out_ch)
        self.lr = nn.LeakyReLU(0.2)
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a_s); nn.init.xavier_uniform_(self.a_d)

    def forward(self, x, ei, ea, ed):
        H = self.W(x).view(-1, self.heads, self.out_ch)
        self._as = (H*self.a_s).sum(-1)
        self._ad = (H*self.a_d).sum(-1)
        self._H = H
        self._eb = self.ep(ea)
        # Additive distance (v2 style — worked better than multiplicative)
        if self.use_dist:
            self._db = (self.gamma * self.dk(ed)).unsqueeze(-1)
        else:
            self._db = torch.zeros(ed.shape[0], 1, device=x.device)
        out = self.propagate(ei, size=None)
        out = out.view(-1, self.heads*self.out_ch) if self.concat else out.mean(1)
        return self.ln(out)

    def message(self, edge_index_i, edge_index_j, size_i):
        a = self._as[edge_index_i] + self._ad[edge_index_j] + self._eb + self._db
        a = softmax(self.lr(a), edge_index_i, num_nodes=size_i)
        a = F.dropout(a, p=self.drop, training=self.training)
        return self._H[edge_index_j] * a.unsqueeze(-1)

    def aggregate(self, inputs, index, dim_size=None):
        from torch_geometric.utils import scatter
        return scatter(inputs, index, dim=0, dim_size=dim_size, reduce='sum')


class StabilityGATv4(nn.Module):
    def __init__(self, n_feat=970, hid=64, heads=4, n_k=3, e_dim=4,
                 drop=0.3, use_dist=True):
        super().__init__()
        self.drop = drop
        self.aa_emb = nn.Embedding(20, 32)
        inp = 32 + (n_feat - 1)  # aa_embed replaces aa_idx
        gd = heads * hid
        self.proj = nn.Sequential(nn.Linear(inp, gd), nn.LayerNorm(gd),
                                  nn.GELU(), nn.Dropout(drop))
        self.g1 = DistGATLayer(gd, hid, heads, n_k, e_dim, drop, True, use_dist)
        self.g2 = DistGATLayer(gd, hid, heads, n_k, e_dim, drop, True, use_dist)
        self.r1 = nn.Linear(gd, gd)
        self.r2 = nn.Linear(gd, gd)

        # Readout: mutation node (gd) + global mean (gd) = 2*gd
        self.reg = nn.Sequential(
            nn.Linear(gd*2, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(drop),
            nn.Linear(256, 64), nn.GELU(), nn.Dropout(drop*0.5), nn.Linear(64, 1))

    def forward(self, data):
        x, ei, ea, ed = data.x, data.edge_index, data.edge_attr, data.edge_dist
        batch, mi = data.batch, data.mut_idx
        aa = self.aa_emb(x[:,0].long().clamp(0,19))
        x = torch.cat([aa, x[:,1:]], 1)
        x = self.proj(x)
        x = F.gelu(self.g1(x, ei, ea, ed) + self.r1(x))
        x = F.dropout(x, self.drop, self.training)
        x = F.gelu(self.g2(x, ei, ea, ed) + self.r2(x))
        x = F.dropout(x, self.drop, self.training)
        # Mutation site + global
        bo = torch.zeros_like(mi)
        if batch is not None:
            for i in range(1, len(mi)): bo[i] = (batch < i).sum()
        mr = x[mi.squeeze(-1) + bo]
        gr = global_mean_pool(x, batch)
        return self.reg(torch.cat([mr, gr], 1)).squeeze(-1)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

model = StabilityGATv4(n_feat=in_ch, use_dist=True).to(device)
print(f"Parameters: {model.count_parameters():,}")
print("✅ Model ready!")

In [ ]:
# Cell 6: Training with Protein-Weighted Loss + Correlation Penalty

CFG = {'epochs': 150, 'bs': 32, 'lr': 2e-4, 'wd': 5e-4, 'patience': 25}


def pearson_loss(pred, target):
    """Differentiable Pearson correlation loss. Minimizing this maximizes r."""
    vp = pred - pred.mean()
    vt = target - target.mean()
    r = (vp * vt).sum() / (torch.sqrt((vp**2).sum()) * torch.sqrt((vt**2).sum()) + 1e-8)
    return 1.0 - r  # minimize 1-r to maximize r


def weighted_huber_loss(pred, target, weights, delta=2.0):
    """Huber loss weighted by protein sample weights."""
    diff = torch.abs(pred - target)
    huber = torch.where(diff < delta, 0.5 * diff**2, delta * (diff - 0.5 * delta))
    return (huber * weights).mean()


def train_ep(model, loader, opt, dev):
    model.train(); tl, n = 0, 0
    for batch in loader:
        batch = batch.to(dev); opt.zero_grad()
        pred = model(batch)
        target = batch.y.squeeze()

        # FIX 1: Protein-weighted loss
        weights = batch.sample_weight if hasattr(batch, 'sample_weight') else torch.ones_like(target)

        # FIX 4: Combined loss = weighted Huber + correlation penalty
        loss_huber = weighted_huber_loss(pred, target, weights, delta=2.0)
        loss_corr = pearson_loss(pred, target) if len(target) > 4 else torch.tensor(0.0, device=dev)
        loss = loss_huber + 0.5 * loss_corr

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += loss.item() * len(target); n += len(target)
    return tl / n


@torch.no_grad()
def ev(model, loader, dev):
    model.eval(); P, T = [], []
    for b in loader:
        b = b.to(dev)
        P.extend(model(b).cpu().numpy())
        T.extend(b.y.squeeze().cpu().numpy())
    P, T = np.array(P), np.array(T)
    return {'pcc': pearsonr(T,P)[0], 'scc': spearmanr(T,P)[0],
            'rmse': np.sqrt(mean_squared_error(T,P)),
            'mae': mean_absolute_error(T,P), 'preds': P, 'targets': T}


def run(name, tr_graphs, val_graphs, te_graphs, use_dist=True):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    trl = DataLoader(tr_graphs, batch_size=CFG['bs'], shuffle=True, drop_last=True)
    vl = DataLoader(val_graphs, batch_size=CFG['bs'])
    tel = DataLoader(te_graphs, batch_size=CFG['bs'])

    m = StabilityGATv4(n_feat=tr_graphs[0].x.shape[1], use_dist=use_dist).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, 30, 2, 1e-6)

    best, pc, bs = -1, 0, None; t0 = time.time()
    for ep in range(CFG['epochs']):
        tl = train_ep(m, trl, opt, device)
        vm = ev(m, vl, device)
        sch.step()
        if vm['pcc'] > best:
            best = vm['pcc']; pc = 0
            bs = {k:v.cpu().clone() for k,v in m.state_dict().items()}
        else:
            pc += 1
            if pc >= CFG['patience']: print(f"  Early stop ep {ep+1}"); break
        if (ep+1) % 10 == 0:
            print(f"  Ep {ep+1}: loss={tl:.4f} val_r={vm['pcc']:.4f} val_rmse={vm['rmse']:.3f}")

    m.load_state_dict(bs); m.to(device)
    r = ev(m, tel, device)
    elapsed = (time.time()-t0)/60
    print(f"\n  S669 Test: r={r['pcc']:.4f} ρ={r['scc']:.4f} RMSE={r['rmse']:.3f} MAE={r['mae']:.3f} ({elapsed:.1f}min)")
    return {**r, 'name': name, 'state': bs, 'time': elapsed}


print("✅ Training functions with all fixes ready!")
print("  - Protein-weighted Huber loss")
print("  - Pearson correlation penalty")
print("  - Protein-level validation (no leak)")

In [ ]:
# Cell 7: Run Experiments

R = {}

# Full model
R['Full'] = run('v4: Distance-Aware GAT (all fixes)',
                actual_train, val_graphs, test_graphs, use_dist=True)

# Ablation: no distance kernel
R['NoDist'] = run('v4: Standard GAT (no distance)',
                  actual_train, val_graphs, test_graphs, use_dist=False)

# Print results
print(f"\n\n{'='*75}")
print(f"S669 BLIND TEST RESULTS (v4)")
print(f"{'='*75}")
print(f"{'Method':<45} {'r':>8} {'ρ':>8} {'RMSE':>8} {'MAE':>8}")
print(f"{'─'*75}")
for n,r in R.items():
    print(f"{r['name']:<45} {r['pcc']:>8.4f} {r['scc']:>8.4f} {r['rmse']:>8.3f} {r['mae']:>8.3f}")
print(f"{'─'*75}")
for nm,rr,rm in [('DynaMut',0.415,1.596),('DDMut',0.440,1.492),
                  ('ACDC-NN',0.460,1.488),('GeoDDG-3D',0.601,1.332)]:
    print(f"{nm+' (published S669)':<45} {rr:>8.3f} {'—':>8} {rm:>8.3f} {'—':>8}")
print(f"{'='*75}")

dp = R['Full']['pcc'] - R['NoDist']['pcc']
print(f"\nDistance kernel Δr = {dp:+.4f}")
print(f"Full vs DynaMut: {'✅ BETTER' if R['Full']['pcc']>0.415 else '⚠️ below'}")
print(f"Full vs DDMut: {'✅ BETTER' if R['Full']['pcc']>0.440 else '⚠️ below'}")

In [ ]:
# Cell 8: Visualization + Save

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

r = R['Full']
ax = axes[0]
ax.scatter(r['targets'], r['preds'], alpha=0.4, s=15, c='steelblue', edgecolors='none')
lims = [min(r['targets'].min(),r['preds'].min())-1, max(r['targets'].max(),r['preds'].max())+1]
ax.plot(lims, lims, 'r--', lw=1, alpha=.5)
ax.set_xlabel('Experimental ΔΔG (kcal/mol)', fontsize=12)
ax.set_ylabel('Predicted ΔΔG (kcal/mol)', fontsize=12)
ax.set_title(f'v4 Full Model: r={r["pcc"]:.3f}, RMSE={r["rmse"]:.2f}', fontsize=13, fontweight='bold')
ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect('equal'); ax.grid(True, alpha=.3)

ax = axes[1]
all_m = list(R.keys()) + ['DynaMut*','DDMut*','ACDC*','GeoDDG*']
all_r = [R[n]['pcc'] for n in R] + [0.415, 0.440, 0.460, 0.601]
colors = ['#2ecc71','#e74c3c'] + ['#95a5a6']*4
names = ['v4 Full','v4 NoDist','DynaMut*','DDMut*','ACDC*','GeoDDG*']
bars = ax.bar(names, all_r, color=colors, edgecolor='black', lw=.5, alpha=.85)
for b,v in zip(bars, all_r): ax.text(b.get_x()+b.get_width()/2, b.get_height()+.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Pearson r'); ax.set_title('S669 Benchmark Comparison', fontweight='bold')
ax.set_ylim(0, .8); ax.grid(True, alpha=.3, axis='y')

ax = axes[2]
model.load_state_dict(R['Full']['state']); model.eval()
for li,(ln,ly) in enumerate([('L1',model.g1),('L2',model.g2)]):
    if hasattr(ly,'dk'):
        dk=ly.dk; s=torch.exp(dk.log_s).detach().cpu().numpy()
        w=F.softmax(dk.w,0).detach().cpu().numpy()
        d=np.linspace(0,1,200); da=d*12
        labels=['H-bond\n(3-4Å)','vdW\n(5-7Å)','Electrostatic\n(8-12Å)']
        clrs=['#e74c3c','#3498db','#2ecc71']
        for m_k in range(dk.n_k):
            resp=w[m_k]*np.exp(-d**2/(2*s[m_k]**2+1e-8))
            lbl=f'{ln}: {labels[m_k]}\nσ={s[m_k]*12:.1f}Å w={w[m_k]:.2f}' if li==0 else None
            ax.plot(da, resp, '-' if li==0 else '--', color=clrs[m_k], lw=2, alpha=.7, label=lbl)
ax.set_xlabel('Distance (Å)', fontsize=12); ax.set_ylabel('Attention Bias', fontsize=12)
ax.set_title('Learned Interaction Ranges', fontsize=13, fontweight='bold')
ax.legend(fontsize=7); ax.grid(True, alpha=.3); ax.set_xlim(0,12)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'results_v4.png'), dpi=150, bbox_inches='tight')
plt.show()

with open(os.path.join(RESULTS_DIR,'results_v4.json'),'w') as f:
    json.dump({n:{k:float(v) for k,v in r.items() if k in ['pcc','scc','rmse','mae']}
               for n,r in R.items()}, f, indent=2)
print(f"Saved to {RESULTS_DIR}")